# Full Dataset Feature Build

This notebook creates a local optimized feature copy of OpenMementos.

The Hugging Face dataset is streamed once, parsed into trace-level and block-level feature tables, and saved as local parquet files under `data/`. The `data/` directory is ignored by Git, so these generated files are not committed.

The output is partitioned into multiple parquet files to avoid holding the full dataset in memory.

In [ ]:
from datetime import datetime
from itertools import islice
from pathlib import Path
import re

import numpy as np
import pandas as pd
import tiktoken
from datasets import load_dataset
from tqdm.auto import tqdm

In [11]:
DATASET_ID = "microsoft/OpenMementos"
SPLIT = "train"
ENCODING_NAME = "cl100k_base"

# Use a small value for testing, then set to None for the full dataset.
#MAX_ROWS = 1_000

# Number of original traces processed before writing one parquet partition.
CHUNK_SIZE = 10_000

# Create a run-specific output directory so full builds do not overwrite earlier runs.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("../data/full_feature_builds") / RUN_ID
TRACE_DIR = OUTPUT_DIR / "traces"
BLOCK_DIR = OUTPUT_DIR / "blocks"

TRACE_DIR.mkdir(parents=True, exist_ok=True)
BLOCK_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

PosixPath('../data/full_feature_builds/20260612_222038')

In [12]:
BLOCK_RE = re.compile(r"<\|block_start\|>(.*?)<\|block_end\|>", re.DOTALL)
SUMMARY_RE = re.compile(r"<\|summary_start\|>(.*?)<\|summary_end\|>", re.DOTALL)
THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)

encoding = tiktoken.get_encoding(ENCODING_NAME)

def count_tokens(text: str) -> int:
    if text is None:
        return 0
    return len(encoding.encode(text))

def count_tokens_batch(texts: list[str]) -> list[int]:
    # Batch tokenization is much faster than encoding each block separately.
    texts = ["" if x is None else x for x in texts]
    return [len(tokens) for tokens in encoding.encode_batch(texts)]

In [13]:
def parse_response(response: str) -> dict:
    # Missing responses are treated as empty strings so parsing remains stable.
    if response is None:
        response = ""

    # The reasoning chain is expected inside one <think>...</think> section.
    think_match = THINK_RE.search(response)

    if think_match:
        think_text = think_match.group(1)
        answer_text = response[think_match.end():].strip()
    else:
        think_text = ""
        answer_text = response.strip()

    # Extract aligned reasoning blocks and summaries from the thinking section.
    blocks = [x.strip() for x in BLOCK_RE.findall(think_text)]
    summaries = [x.strip() for x in SUMMARY_RE.findall(think_text)]

    return {
        "think_text": think_text,
        "answer_text": answer_text,
        "blocks": blocks,
        "summaries": summaries,
        "n_blocks": len(blocks),
        "n_summaries": len(summaries),
        "think_chars": len(think_text),
        "answer_chars": len(answer_text),
        "response_chars": len(response),
    }

In [14]:
MATH_SYMBOLS = set("+-=*/^<>≤≥≈≠√∑∫π%()[]{}")

def extract_problem_features(problem: str) -> dict:
    # Problem features are valid predictors because they are available before
    # observing the generated reasoning blocks or summaries.
    if problem is None:
        problem = ""

    n_chars = len(problem)
    n_math_symbols = sum(ch in MATH_SYMBOLS for ch in problem)

    return {
        "problem_chars": n_chars,
        "problem_tokens": count_tokens(problem),
        "problem_math_symbol_share": n_math_symbols / n_chars if n_chars else 0,
        "problem_has_multiple_choice": int(bool(re.search(r"\b[A-D][\).]", problem))),
        "problem_has_code_fence": int("```" in problem),
        "problem_question_mark_count": problem.count("?"),
    }

In [15]:
def build_feature_chunk(rows: list[dict], start_trace_id: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    trace_rows = []
    block_rows = []

    for offset, row in enumerate(rows):
        trace_id = start_trace_id + offset

        problem = row.get("problem") or ""
        response = row.get("response") or ""

        parsed = parse_response(response)
        problem_features = extract_problem_features(problem)

        think_tokens = count_tokens(parsed["think_text"])
        answer_tokens = count_tokens(parsed["answer_text"])
        response_tokens = count_tokens(response)

        trace_rows.append({
            "trace_id": trace_id,
            "domain": row.get("domain"),
            "source": row.get("source"),
            "difficulty": row.get("difficulty"),
            **problem_features,
            "response_chars": parsed["response_chars"],
            "response_tokens": response_tokens,
            "think_chars": parsed["think_chars"],
            "think_tokens": think_tokens,
            "answer_chars": parsed["answer_chars"],
            "answer_tokens": answer_tokens,
            "n_blocks": parsed["n_blocks"],
            "n_summaries": parsed["n_summaries"],
            "block_summary_delta": parsed["n_blocks"] - parsed["n_summaries"],
        })

        # Tokenize all blocks and summaries from this trace in batches.
        block_token_counts = count_tokens_batch(parsed["blocks"])
        summary_token_counts = count_tokens_batch(parsed["summaries"])

        for block_index, (block, summary, block_tokens, summary_tokens) in enumerate(
            zip(parsed["blocks"], parsed["summaries"], block_token_counts, summary_token_counts)
        ):
            block_chars = len(block)
            summary_chars = len(summary)

            block_rows.append({
                "trace_id": trace_id,
                "block_index": block_index,
                "domain": row.get("domain"),
                "source": row.get("source"),
                "difficulty": row.get("difficulty"),
                **problem_features,
                "n_blocks_in_trace": parsed["n_blocks"],
                "relative_block_position": block_index / (parsed["n_blocks"] - 1) if parsed["n_blocks"] > 1 else 0,
                "block_chars": block_chars,
                "summary_chars": summary_chars,
                "block_tokens": block_tokens,
                "summary_tokens": summary_tokens,
                "summary_to_block_char_ratio": summary_chars / block_chars if block_chars else np.nan,
                "summary_to_block_token_ratio": summary_tokens / block_tokens if block_tokens else np.nan,
                "char_compression_savings": 1 - (summary_chars / block_chars) if block_chars else np.nan,
                "token_compression_savings": 1 - (summary_tokens / block_tokens) if block_tokens else np.nan,
            })

    return pd.DataFrame(trace_rows), pd.DataFrame(block_rows)

In [16]:
def write_feature_partitions(max_rows: int | None = None) -> dict:
    ds_stream = load_dataset(DATASET_ID, split=SPLIT, streaming=True)

    current_chunk = []
    trace_id = 0
    part_id = 0
    total_blocks = 0

    iterator = islice(ds_stream, max_rows) if max_rows is not None else ds_stream

    for row in tqdm(iterator, desc="Streaming traces"):
        current_chunk.append(row)

        # Once the chunk reaches CHUNK_SIZE, convert it to feature tables and
        # write parquet files before continuing. This keeps memory bounded.
        if len(current_chunk) >= CHUNK_SIZE:
            df_traces_chunk, df_blocks_chunk = build_feature_chunk(current_chunk, trace_id)

            df_traces_chunk.to_parquet(TRACE_DIR / f"traces_part_{part_id:04d}.parquet", index=False)
            df_blocks_chunk.to_parquet(BLOCK_DIR / f"blocks_part_{part_id:04d}.parquet", index=False)

            trace_id += len(df_traces_chunk)
            total_blocks += len(df_blocks_chunk)
            part_id += 1
            current_chunk = []

    # Write the final partial chunk, if any rows remain.
    if current_chunk:
        df_traces_chunk, df_blocks_chunk = build_feature_chunk(current_chunk, trace_id)

        df_traces_chunk.to_parquet(TRACE_DIR / f"traces_part_{part_id:04d}.parquet", index=False)
        df_blocks_chunk.to_parquet(BLOCK_DIR / f"blocks_part_{part_id:04d}.parquet", index=False)

        trace_id += len(df_traces_chunk)
        total_blocks += len(df_blocks_chunk)
        part_id += 1

    return {
        "output_dir": str(OUTPUT_DIR),
        "n_trace_rows": trace_id,
        "n_block_rows": total_blocks,
        "n_parts": part_id,
    }

In [17]:
build_summary = write_feature_partitions(max_rows=MAX_ROWS)
build_summary

Streaming traces: 228557it [2:13:58, 28.43it/s] 


{'output_dir': '../data/full_feature_builds/20260612_222038',
 'n_trace_rows': 228557,
 'n_block_rows': 2013510,
 'n_parts': 23}

In [18]:
df_traces_full_test = pd.read_parquet(TRACE_DIR)
df_blocks_full_test = pd.read_parquet(BLOCK_DIR)

df_traces_full_test.shape, df_blocks_full_test.shape

((228557, 19), (2013510, 21))

In [10]:
MAX_ROWS = None
# rerun from block #2

In [19]:
# df blocks 
df_blocks_full = pd.read_parquet(BLOCK_DIR)

# Define the high-compression target globally on the full local feature build.
compression_threshold = df_blocks_full["summary_to_block_token_ratio"].quantile(0.25)

df_blocks_full["high_token_compression"] = (
    df_blocks_full["summary_to_block_token_ratio"] <= compression_threshold
).astype(int)

compression_threshold, df_blocks_full["high_token_compression"].value_counts(normalize=True).round(4)

(np.float64(0.10891089108910891),
 high_token_compression
 0    0.75
 1    0.25
 Name: proportion, dtype: float64)

In [20]:
# Save a labeled block-level table for modeling notebooks.
df_blocks_full.to_parquet(OUTPUT_DIR / "blocks_features_full_labeled.parquet", index=False)

In [24]:
# df traces
df_traces_full = pd.read_parquet(TRACE_DIR)

# Aggregate block-level compression behavior to the trace level. This creates
# one compression summary row per original reasoning trace.
trace_compression = (
    df_blocks_full
    .groupby("trace_id")
    .agg(
        trace_block_tokens=("block_tokens", "sum"),
        trace_summary_tokens=("summary_tokens", "sum"),
        trace_mean_summary_to_block_token_ratio=("summary_to_block_token_ratio", "mean"),
        trace_median_summary_to_block_token_ratio=("summary_to_block_token_ratio", "median"),
        trace_high_compression_share=("high_token_compression", "mean"),
    )
    .reset_index()
)

# The total trace compression ratio compares all summary tokens against all
# original block tokens within the same trace.
trace_compression["trace_total_summary_to_block_token_ratio"] = (
    trace_compression["trace_summary_tokens"]
    / trace_compression["trace_block_tokens"]
)

trace_compression.head()

,trace_id,trace_block_tokens,trace_summary_tokens,trace_mean_summary_to_block_token_ratio,trace_median_summary_to_block_token_ratio,trace_high_compression_share,trace_total_summary_to_block_token_ratio
0,0,3267,898,0.297850,0.303896,0.00,0.274870
1,1,2110,712,0.335386,0.270494,0.00,0.337441
2,2,14102,982,0.132226,0.056338,0.75,0.069636
3,3,5021,733,0.205201,0.104587,0.60,0.145987
4,4,3305,1077,0.350994,0.274430,0.00,0.325870


In [25]:
# Merge trace-level compression summaries onto the original trace-level table.
df_traces_full = df_traces_full.merge(
    trace_compression,
    on="trace_id",
    how="left",
)

df_traces_full.shape

(228557, 25)

In [26]:
# Define a trace-level high-compression target using the lowest quartile of the
# total trace compression ratio.
trace_compression_threshold = (
    df_traces_full["trace_total_summary_to_block_token_ratio"].quantile(0.25)
)

df_traces_full["trace_high_token_compression"] = (
    df_traces_full["trace_total_summary_to_block_token_ratio"] <= trace_compression_threshold
).astype(int)

trace_compression_threshold, df_traces_full["trace_high_token_compression"].value_counts(normalize=True).round(4)

(np.float64(0.1327609995400889),
 trace_high_token_compression
 0    0.75
 1    0.25
 Name: proportion, dtype: float64)

In [27]:
# Save labeled feature tables for downstream full-data modeling notebooks.
df_blocks_full.to_parquet(
    OUTPUT_DIR / "blocks_features_full_labeled.parquet",
    index=False,
)

df_traces_full.to_parquet(
    OUTPUT_DIR / "traces_features_full_labeled.parquet",
    index=False,
)

In [28]:
pd.read_parquet(OUTPUT_DIR / "blocks_features_full_labeled.parquet").shape

(2013510, 22)

In [29]:
pd.read_parquet(OUTPUT_DIR / "traces_features_full_labeled.parquet").shape

(228557, 26)

## Full Feature Build Outputs

The full OpenMementos feature build produced two local labeled parquet files:

- `blocks_features_full_labeled.parquet`: block-level features and the block-level `high_token_compression` target.
- `traces_features_full_labeled.parquet`: trace-level features and the trace-level `trace_high_token_compression` target.

The block-level table has one row per `(reasoning block, summary)` pair. The trace-level table has one row per original reasoning trace.

These files are saved under the ignored `data/` directory and are not committed to Git.